# MaleCNS Connectome to 3D Humanoid Physics Simulation (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gakpey/MaleCNS_human/blob/main/notebooks/02_malecns_humanoid_simulation.ipynb)

This interactive notebook simulates a **3D Humanoid in a physics-based world** controlled by the **MaleCNS (*Drosophila melanogaster*) connectome**. Sensory stimuli propagate through MaleCNS interneurons to descending motor output neurons, driving human body movement.

> **Live Streaming Feature**: Watch the 3D simulation happen **live** inside your Google Colab browser cell without requiring any local GPU hardware!

## Step 1: Environment & Package Installation in Google Colab

In [ ]:
import sys
import os
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("🚀 Initializing Google Colab Runtime...")
    
    # 1. Clone repository if running in a fresh Google Colab session
    repo_dir = Path("/content/MaleCNS_human")
    if not repo_dir.exists() and not Path("src/malecns").exists():
        print("📥 Cloning repository to /content/MaleCNS_human...")
        !git clone https://github.com/Gakpey/MaleCNS_human.git /content/MaleCNS_human
        
    if repo_dir.exists():
        os.chdir(repo_dir)
        print(f"✔ Working directory: {os.getcwd()}")
        
    # 2. Install binary dependencies (fast wheel install)
    !pip install -q pybullet networkx meshcat imageio imageio-ffmpeg mediapy
else:
    print("💻 Running in Local Development Environment")

# 3. Instant direct sys.path import (bypasses slow pip metadata build steps)
possible_paths = [
    Path.cwd() / "src",
    Path.cwd().parent / "src",
    Path("/content/MaleCNS_human/src"),
    Path("/content/src")
]
for p in possible_paths:
    if p.exists() and str(p.resolve()) not in sys.path:
        sys.path.insert(0, str(p.resolve()))
        print(f"✔ Added {p.resolve()} to sys.path")

## Step 2: Initialize MaleCNS Connectome & Physics Engine

In [ ]:
from malecns.config import setup_environment
from malecns.utils import set_seed
from malecns.connectome import MaleCNSNetwork
from malecns.mapping import ConnectomeToHumanoidMapper
from malecns.simulation import HumanoidPhysicsSim
from malecns.render import LiveWebGLViewer, VideoRenderer

# Set seed for reproducibility
set_seed(42)

# Setup paths & configuration
env_paths = setup_environment()

# 1. Load MaleCNS Connectome Network
connectome = MaleCNSNetwork(num_sensory=16, num_interneurons=64, num_descending=16, seed=42)
print("🧠 MaleCNS Connectome Summary:", connectome.get_summary())

# 2. Initialize Connectome-to-Humanoid Motor Mapper
mapper = ConnectomeToHumanoidMapper(num_descending_neurons=16, gain=1.2, smoothing=0.15)
print(f"🦴 Mapped {len(mapper.get_joint_names())} 3D Humanoid Body Degrees of Freedom (DoFs)")

# 3. Initialize Headless 3D Physics Simulator
sim = HumanoidPhysicsSim(render_mode="DIRECT")

## Step 3: Setup Live WebGL 3D Viewer
Run the cell below to initialize the live Three.js 3D WebGL canvas directly inside your Colab cell.

In [ ]:
# Create Live Viewer
live_viewer = LiveWebGLViewer(width=640, height=400)
live_viewer.init_colab_display()

## Step 4: Run Connectome-Driven Physics Simulation
This loop propagates sensory signals through MaleCNS neurons, maps output firing rates to the 3D Humanoid body, steps the physics world, updates the **Live 3D Stream**, and captures frames for MP4 video export.

In [ ]:
import numpy as np
import time

sim_steps = 150
dt = 0.02
captured_frames = []

print("🎬 Starting Connectome-Driven Physics Simulation...")

for step in range(sim_steps):
    # 1. Synthesize dynamic sensory stimulation (visual tracking / mechanosensory input waves)
    t = step * dt
    sensory_stimulus = np.zeros(16)
    sensory_stimulus[0:4] = 0.8 * np.sin(2.0 * np.pi * 0.5 * t) + 0.2 # Visual tracking
    sensory_stimulus[4:8] = 0.6 * np.cos(2.0 * np.pi * 0.8 * t)       # Mechanosensory
    sensory_stimulus[8:12] = 0.9 * (1.0 if (step // 20) % 2 == 0 else 0.1) # Rhythmic walking pulse
    
    # 2. Step MaleCNS neural dynamics forward
    dn_firing_rates = connectome.step(sensory_stimulus, dt=dt)
    
    # 3. Map descending neuron output rates to 3D Humanoid target joint angles
    target_joint_angles = mapper.map_dn_to_joints(dn_firing_rates)
    
    # 4. Apply joint target position motors & step physics world
    sim.apply_joint_targets(target_joint_angles)
    sim.step()
    
    # 5. Read humanoid 3D state
    state = sim.get_state()
    pos = state["position"]
    
    # 6. Push position & joint updates to Live WebGL Viewer
    if step % 2 == 0:
        live_viewer.update(position=pos, joint_angles=target_joint_angles, step=step)
        
    # 7. Render frame for MP4 video export
    rgb_frame = sim.render_frame(width=640, height=480)
    captured_frames.append(rgb_frame)
    
    if (step + 1) % 30 == 0:
        print(f"   Step {step+1}/{sim_steps} complete. Humanoid Height: {pos[2]:.2f}m")

sim.close()
print("✅ Physics Simulation Complete!")

## Step 5: Render & Play Simulation MP4 Video
Compiles the captured physics simulation frames into a high-quality MP4 video file and plays it in the cell below.

In [ ]:
output_video_path = env_paths["results"] / "malecns_humanoid_behavior.mp4"
VideoRenderer.save_video(captured_frames, str(output_video_path), fps=30)

# Display MP4 video player in notebook output
VideoRenderer.display_video_in_colab(str(output_video_path), width=640)